# ETL — V1DD (release 1196) exploration

Loads every artifact in `/data/v1dd_1196/` and prints shape, columns, and a small head for each. Each load is followed by a short note proposing which common-connectivity schema the file maps to. No writes — this notebook is a planning aid for the subsequent `etl_v1dd_01_*`, `etl_v1dd_02_*`, etc. notebooks.

V1DD is the same modality as MICrONS Minnie (EM connectomics + coregistered 2P functional imaging). Schema mapping mirrors the `etl_minnie_*` notebooks where applicable.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
DATA_ROOT = Path("/data/v1dd_1196")
PROJECT_ID = "v1dd"
RELEASE = "1196"

print(f"DATA_ROOT  : {DATA_ROOT}")
print(f"PROJECT_ID : {PROJECT_ID}")
print(f"RELEASE    : {RELEASE}")
print()
print("Contents:")
for p in sorted(DATA_ROOT.iterdir()):
    print("  -", p.name)

DATA_ROOT  : /data/v1dd_1196
PROJECT_ID : v1dd
RELEASE    : 1196

Contents:
  - cell_cell_correlations_by_stimulus.feather
  - cell_cell_correlations_by_stimulus_coregistered.feather
  - coregistration_1196.feather
  - data_description.json
  - metadata.nd.json
  - original_metadata
  - proofread_axon_list_1196.npy
  - proofread_dendrite_list_1196.npy
  - snr_by_cell.feather
  - soma_and_cell_type_1196.feather
  - subject.json
  - syn_df_all_to_proofread_to_all_1196.feather
  - syn_label_df_all_to_proofread_to_all_1196.feather


## 1. Provenance metadata (JSON)

`data_description.json`, `subject.json`, `metadata.nd.json` are aind-data-schema records that describe the release as a whole.

In [3]:
data_desc = json.loads((DATA_ROOT / "data_description.json").read_text())
subject = json.loads((DATA_ROOT / "subject.json").read_text())

print("name           :", data_desc["name"])
print("project_name   :", data_desc["project_name"])
print("modalities     :", [m["abbreviation"] for m in data_desc["modalities"]])
print("institution    :", data_desc["institution"]["abbreviation"])
print("license        :", data_desc["license"])
print("subject_id     :", data_desc["subject_id"])
print("genotype       :", subject["subject_details"]["genotype"])
print("sex            :", subject["subject_details"]["sex"])
print("species        :", subject["subject_details"]["species"]["common_name"])
print("S3 location    :", json.loads((DATA_ROOT / "metadata.nd.json").read_text())["location"])

name           : v1dd-analysis-1196-1_2025-08-14_16-38-00
project_name   : V1 Deep Dive
modalities     : ['EM']
institution    : AIBS
license        : CC-BY-4.0
subject_id     : 409828
genotype       : Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-GCaMP6s)/wt
sex            : Male
species        : House mouse


S3 location    : s3://aind-open-data/v1dd-analysis-1196-1_2025-08-14_16-38-00


**Schema mapping:** `core_schema.yaml::DataSet` — exactly one row, `project_id="v1dd"`, modality `ELECTRON_MICROSCOPY`. The `publication` slot can hold the V1DD release reference; `name` comes from `data_description.name`. Subject metadata (genotype, sex, species) has no slot in the current core schema and would be dropped or recorded only in `DataSet.name`/notes.

## 2. EM soma table — `soma_and_cell_type_1196.feather`

The catalog of EM somas detected in the V1DD volume. Direct analogue of MICrONS' `nucleus_detection_lookup_v1` CAVE view used by `etl_minnie_01_dataset_dataitem.ipynb`.

In [4]:
soma_df = pd.read_feather(DATA_ROOT / "soma_and_cell_type_1196.feather")
print("shape  :", soma_df.shape)
print("cols   :", list(soma_df.columns))
print("dtypes :\n", soma_df.dtypes)
print("n_unique pt_root_id :", soma_df['pt_root_id'].nunique())
print("n_unique id         :", soma_df['id'].nunique())
print("cell_type_coarse counts:\n", soma_df['cell_type_coarse'].value_counts(dropna=False).head())
print("cell_type counts (top 10):\n", soma_df['cell_type'].value_counts(dropna=False).head(10))
soma_df.head(3)

shape  : (207455, 11)
cols   : ['id', 'pt_position_x', 'pt_position_y', 'pt_position_z', 'pt_position_trform_x', 'pt_position_trform_y', 'pt_position_trform_z', 'pt_root_id', 'volume', 'cell_type_coarse', 'cell_type']
dtypes :
 id                        int64
pt_position_x             int64
pt_position_y             int64
pt_position_z             int64
pt_position_trform_x    float64
pt_position_trform_y    float64
pt_position_trform_z    float64
pt_root_id                int64
volume                  float64
cell_type_coarse         object
cell_type                object
dtype: object
n_unique pt_root_id : 163064
n_unique id         : 207455
cell_type_coarse counts:
 cell_type_coarse
None    158263
E        42495
I         6697
Name: count, dtype: int64
cell_type counts (top 10):
 cell_type
None     158263
L6-CT     11260
L4-IT      7955
L3-IT      6361
L6-IT      6044
L5-IT      5090
L2-IT      3073
PTC        2951
L5-ET      2013
DTC        1933
Name: count, dtype: int64


,id,pt_position_x,pt_position_y,pt_position_z,pt_position_trform_x,pt_position_trform_y,pt_position_trform_z,pt_root_id,volume,cell_type_coarse,cell_type
0,228132,632828,749849,738270,-323721.447979,549910.283106,392909.832613,864691132737039043,458.464831,None,None
1,543247,1304922,977915,83880,330339.020171,595962.275760,-306424.551354,864691132730839988,73.345940,None,None
2,203262,624680,531094,283770,-252082.627894,203770.728235,21544.029756,864691132654552792,338.276613,E,L3-IT


**Schema mapping:**

- `core_schema.yaml::DataItem` — one row per nucleus, `id = str(row.id)` (the soma id), `name = str(row.pt_root_id)`. Mirrors `etl_minnie_01`. Also `DataItemDataSetAssociation` linking each soma to the V1DD `DataSet`.
- `cell_features_schema.yaml::CellFeatureMatrix` — `pt_position_{x,y,z}` (voxel-space soma centroid), `pt_position_trform_{x,y,z}` (transformed/CCF coords), and `volume` make a numeric feature set (e.g. `feature_set_id = "v1dd_soma_geometry"`).
- `cell_type_coarse` / `cell_type` — categorical labels. Two options: (a) write as categorical columns inside a `CellFeatureMatrix` (cf. Minnie's CSM coarse types), or (b) treat the V1DD coarse/fine cell-type taxonomy as a `clustering_schema.yaml::ClusterHierarchy` and write `ClusterMembership` rows. Pattern (b) matches `etl_minnie_03_cluster_and_cluster_membership.ipynb`.

## 3. Proofread axon / dendrite lists — `.npy`

Lists of `pt_root_id`s whose axon (resp. dendrite) has been manually proofread. These define the proofread cohort used in the synapse table below.

In [4]:
axon_ids = np.load(DATA_ROOT / "proofread_axon_list_1196.npy", allow_pickle=True)
dend_ids = np.load(DATA_ROOT / "proofread_dendrite_list_1196.npy", allow_pickle=True)

print("axon list      : shape", axon_ids.shape, "dtype", axon_ids.dtype)
print("  n_unique     :", len(set(axon_ids.tolist())))
print("  sample       :", axon_ids[:5].tolist())
print()
print("dendrite list  : shape", dend_ids.shape, "dtype", dend_ids.dtype)
print("  n_unique     :", len(set(dend_ids.tolist())))
print("  sample       :", dend_ids[:5].tolist())
print()
print("overlap axon ∩ dendrite :", len(set(axon_ids.tolist()) & set(dend_ids.tolist())))

axon list      : shape (1210,) dtype int64
  n_unique     : 1210
  sample       : [864691132534275418, 864691132534315610, 864691132535664474, 864691132536286810, 864691132536904794]

dendrite list  : shape (63986,) dtype int64
  n_unique     : 63986
  sample       : [864691132496108732, 864691132511800666, 864691132525163794, 864691132533275738, 864691132533347418]

overlap axon ∩ dendrite : 1148


**Schema mapping:** These are cohort definitions, not features. Best modelled as two extra `core_schema.yaml::DataSet` rows (e.g. `v1dd_1196_proofread_axons`, `v1dd_1196_proofread_dendrites`) with their own `DataItemDataSetAssociation` rows pointing at the existing soma `DataItem` ids. Same pattern as the Minnie cohort DataSets noted in `etl_minnie_01`'s summary cell. Note: ids here are `pt_root_id` (int64); the soma `DataItem`s above are keyed by the soma `id` column — a `pt_root_id → soma_id` join is required before writing the associations.

## 4. Functional coregistration — `coregistration_1196.feather`

Maps EM `pt_root_id`s to functional 2P ROIs (volume / column / plane / roi tuple).

In [7]:
coreg_df = pd.read_feather(DATA_ROOT / "coregistration_1196.feather")
print("shape :", coreg_df.shape)
print("cols  :", list(coreg_df.columns))
print("n_unique pt_root_id :", coreg_df['pt_root_id'].nunique())
print("n_unique (volume,column,plane,roi):", coreg_df.drop_duplicates(['volume','column','plane','roi']).shape[0])
coreg_df.head(3)

shape : (571, 5)
cols  : ['pt_root_id', 'column', 'volume', 'plane', 'roi']
n_unique pt_root_id : 553
n_unique (volume,column,plane,roi): 565


,pt_root_id,column,volume,plane,roi
0,864691132830842994,1,3,0,143
1,864691132741466457,1,3,2,40
2,864691132770893729,1,3,3,98


In [11]:
coreg_df.roi.unique()

array([ 143,   40,   98,  100,   60,  105,   12,  232,  109,  409,  269,
        230,   29,  443,  170,  145,  226,  402,   99,  144,  120,   38,
        206,  117,   21,   30,    4,  341,   22,   25,  361,   14,   19,
          0,  240,  166,  444,  159,  189,  346,   75,  360,  548,  212,
        207,  215,   89,  187,   31,   45,    6,  271,  129,  139,  158,
        237,  245,  112,    5,  177,  367,  197,  193,  318,  150,  163,
         49,   93,  368,  254,  203,  247,   33,   15,   62,   69,   90,
        119,  154,  169,  195,  192,  184,  140,  116,  222,   77,  228,
        191,  121,   94,  141,   10,   36,   52,   32,    3,   67,  108,
         70,   73,   74,   78,   92,   97,  113,   58,  125,  134,  152,
         61,   72,   17,   84,   46,   26,   39,   41,   44,  671,   48,
         43,  227,  457,  127,  122,  229,  176,  107,   87,  231,  380,
        148,  255,  379,  258,  552,  295,  251,  261,  623,  481,   34,
        316,  432,  257,  223,  211,  137,  173,  4

**Schema mapping:** A cross-modal cell-to-cell link table. Two reasonable options:

- `mappings_schema.yaml` — if a `CellToCellMapping` (or similar cross-cell mapping) class exists, this is the natural home (EM cell ↔ functional cell).
- Otherwise, register the coregistered functional cells as `DataItem`s in a `v1dd_coregistered_functional_cells` `DataSet` (id = the 4-tuple stringified), then write association rows. The mapping itself (EM ↔ functional) can be a `CellCellConnectivityLong` row with a relation tag like `coregistration` — but that is a stretch and a dedicated mapping class is preferred. Schema-fit decision deferred to notebook `_03`.

## 5. Functional SNR — `snr_by_cell.feather`

One SNR scalar per functional ROI (keyed by the same `volume / column / plane / roi` tuple as the coregistration table).

In [7]:
snr_df = pd.read_feather(DATA_ROOT / "snr_by_cell.feather")
print("shape :", snr_df.shape)
print("cols  :", list(snr_df.columns))
print("n_unique cells:", snr_df.drop_duplicates(['volume','column','plane','roi']).shape[0])
print("snr describe :\n", snr_df['snr'].describe())
snr_df.head(3)

shape : (4458, 5)
cols  : ['column', 'volume', 'plane', 'roi', 'snr']
n_unique cells: 4458
snr describe :
 count    4458.000000
mean        4.196671
std         4.135927
min         0.953515
25%         2.021927
50%         3.285306
75%         4.877459
max        93.560258
Name: snr, dtype: float64


,column,volume,plane,roi,snr
0,1,3,0,0,2.974124
1,1,3,0,1,2.304902
2,1,3,0,2,1.442091


**Schema mapping:** `cell_features_schema.yaml::CellFeatureMatrix` with one `CellFeatureDefinition` (`snr`, dtype `<f8`). `feature_set_id = "v1dd_functional_snr"`, scoped to the functional-cell `DataItem`s registered from coregistration (§4). Note the row count (4458) is much larger than the coregistered set (571) — most rows are non-coregistered functional cells; those would need their own non-EM-linked DataItem registration if we want to keep them.

## 6. Synapses — `syn_df_all_to_proofread_to_all_1196.feather` (+ labels)

Per-synapse table: `id`, pre/post root ids, pre/post/centroid xyz, `size`. Restricted to synapses touching the proofread cohort. The label feather tags a subset with morphology labels (e.g. `spine`).

In [8]:
syn_df = pd.read_feather(DATA_ROOT / "syn_df_all_to_proofread_to_all_1196.feather")
syn_label_df = pd.read_feather(DATA_ROOT / "syn_label_df_all_to_proofread_to_all_1196.feather")

print("syn_df shape          :", syn_df.shape)
print("syn_df cols           :", list(syn_df.columns))
print("n_unique pre_pt_root  :", syn_df['pre_pt_root_id'].nunique())
print("n_unique post_pt_root :", syn_df['post_pt_root_id'].nunique())
print("size describe         :\n", syn_df['size'].describe())
print()
print("syn_label_df shape    :", syn_label_df.shape)
print("syn_label_df cols     :", list(syn_label_df.columns))
print("syn_label_df index    :", syn_label_df.index.name)
print("tag counts            :\n", syn_label_df['tag'].value_counts(dropna=False).head())
syn_df.head(3)

syn_df shape          : (8204497, 13)
syn_df cols           : ['id', 'pre_pt_position_x', 'pre_pt_position_y', 'pre_pt_position_z', 'post_pt_position_x', 'post_pt_position_y', 'post_pt_position_z', 'ctr_pt_position_x', 'ctr_pt_position_y', 'ctr_pt_position_z', 'size', 'pre_pt_root_id', 'post_pt_root_id']


n_unique pre_pt_root  : 3522176
n_unique post_pt_root : 755702


size describe         :
 count    8.204497e+06
mean     1.235114e+03
std      1.199857e+03
min      1.000000e+02
25%      5.090000e+02
50%      8.770000e+02
75%      1.512000e+03
max      3.239900e+04
Name: size, dtype: float64

syn_label_df shape    : (6706286, 1)
syn_label_df cols     : ['tag']
syn_label_df index    : id
tag counts            :
 tag
shaft    3827760
spine    2690607
soma      187919
Name: count, dtype: int64


,id,pre_pt_position_x,pre_pt_position_y,pre_pt_position_z,post_pt_position_x,post_pt_position_y,post_pt_position_z,ctr_pt_position_x,ctr_pt_position_y,ctr_pt_position_z,size,pre_pt_root_id,post_pt_root_id
0,354386968,758200.5,802316.1,304380.0,757861.0,802558.6,304650.0,757967.7,802597.4,304380.0,240,864691132536286810,864691132734919083
1,378070488,792063.2,514342.5,183735.0,792664.6,514284.3,183915.0,792412.4,514294.0,183735.0,3056,864691132572190492,864691132606767301
2,499493001,977071.3,390075.8,191340.0,976974.3,390104.9,190935.0,976838.5,390337.7,190935.0,1346,864691132573738810,864691132747578447


**Schema mapping:**

- `cell_cell_schema.yaml::CellCellConnectivityLong` — aggregate synapses per (pre, post) pair into synapse-count and total-size weights, written to a dedicated subdirectory per §5g of the prompt guide (e.g. `cellcellconnectivitylong_all_to_proofread_to_all/`). Mirrors `etl_minnie_04_cell_cell.ipynb`.
- Raw per-synapse rows (8.2M) do **not** fit any current schema — there is no per-synapse class in the common schemas. They would either stay as a parquet sidecar or be summarized away. The label table (spine vs other) is per-synapse and would be summarized in the same aggregation (e.g. as `n_spine_synapses` weight or a separate connectivity matrix).

## 7. Functional cell–cell correlations

`cell_cell_correlations_by_stimulus.feather` — all functional ROI pairs, one Pearson correlation per stimulus condition.

`cell_cell_correlations_by_stimulus_coregistered.feather` — same, but restricted to coregistered EM cells and keyed by `pt_root_id` rather than ROI tuple.

In [9]:
corr_df = pd.read_feather(DATA_ROOT / "cell_cell_correlations_by_stimulus.feather")
corr_co_df = pd.read_feather(DATA_ROOT / "cell_cell_correlations_by_stimulus_coregistered.feather")

stim_cols = ['drifting_gratings_full','drifting_gratings_windowed','locally_sparse_noise',
             'natural_images','natural_images_12','natural_movie','spontaneous']

print("corr_df shape           :", corr_df.shape)
print("corr_df cols            :", list(corr_df.columns))
print("stimulus columns        :", stim_cols)
print()
print("corr_co_df shape        :", corr_co_df.shape)
print("corr_co_df cols         :", list(corr_co_df.columns))
print("corr_co_df n_unique pre :", corr_co_df['pre_pt_root_id'].nunique())
print("corr_co_df n_unique post:", corr_co_df['post_pt_root_id'].nunique())
corr_co_df.head(3)

corr_df shape           : (8846260, 13)
corr_df cols            : ['pre_roi', 'post_roi', 'pre_plane', 'post_plane', 'column', 'volume', 'drifting_gratings_full', 'drifting_gratings_windowed', 'locally_sparse_noise', 'natural_images', 'natural_images_12', 'natural_movie', 'spontaneous']
stimulus columns        : ['drifting_gratings_full', 'drifting_gratings_windowed', 'locally_sparse_noise', 'natural_images', 'natural_images_12', 'natural_movie', 'spontaneous']

corr_co_df shape        : (148728, 9)
corr_co_df cols         : ['pre_pt_root_id', 'post_pt_root_id', 'drifting_gratings_full', 'drifting_gratings_windowed', 'locally_sparse_noise', 'natural_images', 'natural_images_12', 'natural_movie', 'spontaneous']
corr_co_df n_unique pre : 551
corr_co_df n_unique post: 551


,pre_pt_root_id,post_pt_root_id,drifting_gratings_full,drifting_gratings_windowed,locally_sparse_noise,natural_images,natural_images_12,natural_movie,spontaneous
0,864691132631872354,864691132993747701,0.050142,0.027819,0.154472,0.181724,0.158909,0.004957,0.078471
1,864691132631872354,864691132786447756,0.055267,0.018284,0.119418,0.115587,0.124021,0.010296,0.197349
2,864691132631872354,864691132617961537,0.065444,0.062367,0.143660,0.065078,0.073297,0.053197,0.108968


**Schema mapping:** Both tables are cell-pair × scalar-per-stimulus → `cell_cell_schema.yaml::CellCellConnectivityLong`, one folder per stimulus condition (§5g pattern) **per table**:

- `cellcellconnectivitylong_func_corr_<stimulus>/` — keyed by functional-cell ids (from §4 registration). 8.8M rows × 7 stimuli ≈ 62M rows total; may want to threshold or sample.
- `cellcellconnectivitylong_func_corr_coreg_<stimulus>/` — keyed by EM `pt_root_id` (i.e. by soma `DataItem` ids). 149k rows per stimulus; small.

The coregistered version is the one with direct anatomical interpretability and should be prioritized.

## Summary — proposed notebook split

| File(s) | Schema target | Future notebook |
|---|---|---|
| `data_description.json`, `subject.json`, `soma_and_cell_type_1196.feather` | `DataSet` + `DataItem` + `DataItemDataSetAssociation` | `etl_v1dd_01_dataset_dataitem.ipynb` |
| `proofread_axon_list_1196.npy`, `proofread_dendrite_list_1196.npy` | extra cohort `DataSet`s + associations | `etl_v1dd_01_dataset_dataitem.ipynb` (or `_01b`) |
| `soma_and_cell_type_1196.feather` (numeric cols), `snr_by_cell.feather` | `CellFeatureMatrix` | `etl_v1dd_02_cell_features.ipynb` |
| `soma_and_cell_type_1196.feather` (`cell_type` / `cell_type_coarse`) | `ClusterHierarchy` + `ClusterMembership` | `etl_v1dd_03_cluster_and_cluster_membership.ipynb` |
| `coregistration_1196.feather` | cross-modal mapping (schema TBD; see §4 note) | `etl_v1dd_03_mapping.ipynb` |
| `syn_df_…_1196.feather` (+ labels) | `CellCellConnectivityLong` (aggregated) | `etl_v1dd_04_cell_cell.ipynb` |
| `cell_cell_correlations_by_stimulus_coregistered.feather` | `CellCellConnectivityLong` (one folder per stimulus) | `etl_v1dd_04_cell_cell.ipynb` |
| `cell_cell_correlations_by_stimulus.feather` | same, functional-cell-keyed | `etl_v1dd_04_cell_cell.ipynb` (defer if functional cells aren't registered) |

**Open questions for the schema owner before writing _01:**
1. Is there a canonical cross-modal cell-link class for the EM↔functional coregistration table, or should it ride on `CellCellConnectivityLong`?
2. Should non-coregistered functional cells (the 4458 in `snr_by_cell` minus the 571 coregistered) be registered as `DataItem`s? If yes, with what id scheme — the `(volume, column, plane, roi)` 4-tuple stringified?
3. Does the V1DD cell-type taxonomy already exist as a `ClusterHierarchy` somewhere (shared with MICrONS CSM), or does this dataset own it?